In [ ]:
import pandas as pd
import numpy as np
import joblib

from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

# Load data
df = pd.read_csv("../data/heart.csv")
df = df.replace("?", np.nan)

# Keep only essential numeric columns (FAST)
use_cols = ["age", "trestbps", "chol", "thalch", "oldpeak", "num"]
df = df[use_cols]

# Target
y = (df["num"] > 0).astype(int)
X = df.drop(columns=["num"])

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Very fast pipeline
model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="liblinear",
        max_iter=300,      # LOWER iterations = faster
        random_state=RANDOM_STATE
    ))
])

model.fit(X_train, y_train)

joblib.dump(model, "../model_pipeline.joblib")

print("Model trained & saved (FAST)")


In [ ]:
import joblib
import numpy as np

# Load saved model
loaded_model = joblib.load("../model_pipeline.joblib")

# Compare predictions
pred_before = model.predict(X_test)
pred_after = loaded_model.predict(X_test)

print("Predictions identical:", np.array_equal(pred_before, pred_after))


In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

model_retrain = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logreg", LogisticRegression(
        solver="liblinear",
        max_iter=300,
        random_state=42
    ))
])

model_retrain.fit(X_train, y_train)

pred_retrain = model_retrain.predict(X_test)

print("Reproducible:", np.array_equal(pred_before, pred_retrain))


In [ ]:
import matplotlib.pyplot as plt

plt.bar(["Before Save", "After Reload"], [pred_before.sum(), pred_after.sum()])
plt.title("Model Reload Prediction Check")
plt.tight_layout()
plt.savefig("../plots/model_reload_check.png")
plt.show()
